# ch04 Bonus 08：跨层 KV 共享（Cross-Layer KV Sharing）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/10_kv-sharing
> **参考真实模型**：LiteLLM / YOCO（You Only Compute Once）/ CLA（Cross-Layer Attention）

## 一句话

让**相邻几层 Transformer 共享同一份 K/V 投影**，而不是每层各算一份。层数越多省得越多，KV 缓存显存大幅下降。

## 动机

标准 Transformer 每层都独立计算自己的 K 和 V（各有 `W_key`、`W_value`）。但研究发现相邻层的 K/V 高度相似——那为什么不共享？

| 方案 | n_layers=12, group_size=3 |
|------|---------------------------|
| 标准每层独立 | 12 组 W_key+W_value |
| **每 3 层共享** | **4 组**，省 67% KV 参数与缓存 |

代价：层间表达能力略有损失，但因相邻层 K/V 本就相似，质量下降可接受。

## 核心结构

把 K/V 投影从「每层一份」改成「每 `group_size` 层一组」。同组内的层共享 K/V，只各自的 `W_query` 和 `out_proj` 独立。

In [ ]:
import torch
import torch.nn as nn


class SharedKVAttention(nn.Module):
    """共享 K/V 的注意力层：K/V 由外部（共享投影）传入，本层只算 query。"""

    def __init__(self, d_in, d_out, num_heads, dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, keys, values):
        b, n, _ = x.shape
        H, hd = self.num_heads, self.head_dim
        q = self.W_query(x).view(b, n, H, hd).transpose(1, 2)
        attn_scores = q @ keys.transpose(2, 3)
        attn_weights = torch.softmax(attn_scores / hd ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = (attn_weights @ values).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(out)


class SharedKVTransformer(nn.Module):
    """演示跨层 KV 共享：每 group_size 层共用一组 W_key/W_value。"""

    def __init__(self, d_in, d_out, context_length, num_heads, n_layers=6,
                 group_size=3, dropout=0.0):
        super().__init__()
        self.group_size = group_size
        self.n_layers = n_layers
        # 每 group_size 层共享一组 K/V 投影
        n_groups = n_layers // group_size
        self.kv_projs = nn.ModuleList([
            nn.Linear(d_in, d_out, bias=False)  # W_key（同时复用为 W_value 的示意）
            for _ in range(n_groups * 2)        # 每组 2 个：W_key + W_value
        ])
        # 每层独立的 query 注意力
        self.attns = nn.ModuleList([
            SharedKVAttention(d_in, d_out, num_heads, dropout) for _ in range(n_layers)
        ])

    def forward(self, x):
        b, n, d = x.shape
        outs = []
        for i, attn in enumerate(self.attns):
            grp = i // self.group_size
            k = self.kv_projs[grp * 2](x).view(b, n, attn.num_heads, attn.head_dim).transpose(1, 2)
            v = self.kv_projs[grp * 2 + 1](x).view(b, n, attn.num_heads, attn.head_dim).transpose(1, 2)
            outs.append(attn(x, k, v))
        return outs

## 2. 运行并对比参数量

In [ ]:
torch.manual_seed(123)
batch, seq, dim, n_heads = 2, 8, 768, 12
x = torch.randn(batch, seq, dim)

model = SharedKVTransformer(dim, dim, 1024, n_heads, n_layers=6, group_size=3)
outs = model(x)
print(f"{model.n_layers} 层输出，每层形状: {tuple(outs[0].shape)}")

# 对比 KV 投影参数
n_groups = model.n_layers // model.group_size
kv_shared = n_groups * 2 * dim * dim       # 每组 W_key+W_value
kv_separate = model.n_layers * 2 * dim * dim
print(f"\n6 层、每 3 层共享一组：")
print(f"  共享方案: {n_groups} 组 KV 投影 = {kv_shared:,} 参数")
print(f"  独立方案: {model.n_layers} 组 KV 投影 = {kv_separate:,} 参数")
print(f"  省: {100 * (1 - kv_shared / kv_separate):.0f}%")
print(f"\n💡 同样的省法也适用于推理 KV 缓存：只需缓存 n_groups 份而非 n_layers 份。")

---
> 📌 本 notebook 实现跨层 KV 共享结构并验证参数节省。
> YOCO / CLA 的完整实现见官方 `ch04/10_kv-sharing`。